# Phase 1: back half of the registration model with oracle and perturbed seeds

## Description

<!-- Written before the first run (2026-10-23, 11:50). Spec: docs/spec_phase1_experiments.md (user). Criteria drafted by Claude; the spec's [decide] items use its proposed defaults, listed under Limits. -->

**Question.** The registration design (docs/design_2026-10-23_gw-landmarks-displacement-field.md) has a front half that finds candidate correspondences and a back half (steps 3–5) that refines a correspondence field and reads it out. This experiment tests the back half on its own, given oracle or approximate-oracle correspondences, under controlled input (T) and output (h) distortions. Four questions:
1. **What is B worth when transfer is perfect?** The effective-data curve (E2).
2. **Can the read-out turn the true correspondence into that value?** (E3)
3. **How wrong may a seed be before refinement can no longer recover it?** (E4)
4. **How do T and h differ?**

**Setup.**
- **Prior** (`experimental/regpfn/prior.py`). f_A is a BNN draw (pfn-repro), evaluated raw so it extends beyond [0, 1]^d. B's function is f_B(u) = h(f_A(T(u))), and both tasks sample x ∈ [0, 1]^d (N_B = 256 uniform; A has n_A ∈ [0, 20] context points and 128 queries).
  - **T** = Kumaraswamy warp ∘ per-dim scale (a ∈ logU[0.7, 1.4]) ∘ shift (s ∈ U[−0.3, 0.3]).
  - **h** = affine ∘ sinh-arcsinh warp (δ ∈ logU[0.5, 2], ε ∈ U[−1, 1]).
  - **Normalization and noise.** Each task is normalized by min/max of its noise-free function on a dense grid of its own window, onto [0.1, 0.9]. Noise σ_ε ∈ logU[1e-3, 3e-2] is added afterwards and y is clipped to [0, 1].
  - **Cells:** C00 (neither), C10 (T), C01 (h), C11 (both).
  - **Oracle:** B_inA = (T(u), norm_A(h⁻¹(norm_B⁻¹(y_B)))), masked where T(u) ∉ [0, 1]^d. E0 checks: exact to < 5e-7 without noise in all cells, d = 1, 2 (tests/unit/test_regpfn.py; plot grid in the log).
- **E1/E2: A-only PFN.** pfn-repro's PFN (6 layers, d_model 256, 64 uniform bins on [0, 1]), trained here on A's data from this prior, with context lengths up to 20 + 256. E2: NLL on A's queries with A's n_A points plus m random inside-window B_inA points, m ∈ {0, 1, 2, 4, …, 256}, 5 random subsets. Then m* = the m at which this curve reaches a model's NLL (monotone interpolation, same cell and n_A).
- **Back half** (`RegBack`, `experimental/regpfn/model.py`). One seeded slot + the null slot.
  - **Refinement:** a weight-tied GRU. Each step does a distance-biased lookup of A's points at x̂ and gates the update by evidence (no A point near ⇒ no move). Corrections are smoothed over B's own coordinates.
  - **Read-out:** distance-biased attention in A's frame. Mixture with the null slot through a learned weight head.
  - **Loss:** mixture NLL + LUPI field loss Σ_t 0.8^{T−t} |(x̂_t, ŷ_t) − B_inA|² on inside points.
- **Runs** (d = 1, one each, 25 min of training on ulysses' RTX 3090):
  - `aonly` (E1 + E2);
  - `back true, n_iter = 0` (E3);
  - `back perturb, n_iter = 8` and `n_iter = 0` (E4). Training σ ∈ U[0, 0.25] for x and y, each on with probability ½; 30% of x-perturbations are global shifts.
- **Evaluation.** 128 fixed pairs per cell (the same pairs in every run), n_A ∈ {0, 2, 5, 10, 20}. Perturbation ladder σ ∈ {0.02, 0.05, 0.1, 0.2} for x (smooth field), shift, and y.

**What to look for.**
- E1: NLL vs n_A.
- E2: NLL(m) per cell and n_A. Curves should differ across cells only through coverage (inside fraction) and noise distortion; a large difference means a bug.
- E3a: in C00 with the true seed, does the model reach the m ≈ N_B end of the curve, and never fall below A alone?
- E3b: m* per cell with the true seed and no refinement.
- E4, per cell and kind: m* vs σ; field error before and after refinement vs σ; null weight vs σ; n_iter = 8 vs 0.

**Decision rule** (drafted by Claude; d = 1, n_A ∈ {2, 5, 10, 20}):
- **E3a passes** if in C00 the true-seed model's NLL ≤ the A-only NLL (m = 0) + 0.02 at every n_A, and its m* ≥ 128.
- **E3b: the read-out is OK** if m* ≥ 128 (half of N_B) in all four cells; **WEAK** if 16 ≤ m* < 128 in some cell; **FAIL** if m* < 16 in some cell.
- **E4: refinement helps** in a (cell, kind) if NLL(n_iter = 8) < NLL(n_iter = 0) on perturbed seeds for σ ≥ 0.05 at n_A ≥ 5. The **convergence radius** is the largest σ on the ladder at which refinement cuts the field error by ≥ 50% at n_A = 10. The null weight should rise with σ.
- **Stop rule** (spec §9): if E3a fails, E4 is not interpreted.

**Limits.**
- **[decide] defaults, flagged:**
  - BNN: pfn-repro's own ranges, evaluated raw outside [0, 1]^d.
  - T: s_max 0.3, a ∈ [0.7, 1.4], Kumaraswamy log-sd 0.35.
  - h: sinh-arcsinh plus affine. The affine part is invisible after B's min/max normalization.
  - Normalization: per-task min/max onto [0.1, 0.9], noise added after it, clipped to [0, 1].
  - Bar distribution: uniform bins rather than prior quantiles.
  - Model sizes: back half d_model 128, 3 read-out layers; A-only PFN 6 × 256. T_iter = 8, Fourier features (8 octaves).
- **A_inB** is not produced (not needed for the back half).
- **Training budget:** 25 min per model is short, and m* is read off our own A-only PFN, so the ceiling is only as good as that model.
- **Scope:** d = 1 only in this round. 100% related pairs: no unrelated pairs, so the null slot is never right; its weight only measures distrust of a bad seed.


In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/regpfn/<sweep_id>"  # set after `just snapshot runs/regpfn/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.